## **1. Environment Setup**

In [1]:
# install the required libraries
!pip install -q \
  "transformers==5.0.0" \
  "peft==0.18.1" \
  "accelerate==1.13.0" \
  "datasets==4.8.4" \
  "trl==1.1.0" \
  "sentencepiece==0.2.1" \
  "protobuf==5.29.6" \
  "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.3 MB/s eta 0:00:00


In [2]:
import sys
import torch

print("Python     :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)

# check GPU availability
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name   :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory : {props.total_memory / 1024**3:.2f} GB")

# Expected on Colab free tier: Tesla T4, ~14.5 GB

Python     : 3.12.13
PyTorch      : 2.10.0+cu128
CUDA available : True
GPU name   : Tesla T4
GPU memory : 14.56 GB


In [3]:
EPOCHS=3
TRAIN_EXAMPLES=5000
PER_DEV_TRAIN_BATCH_SIZE=4
PER_DEV_EVAL_BATCH_SIZE=4
GRAD_ACC_STEPS=4
LR=2e-4

## **2. Load the base model**

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-Coder-3B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# For standard LoRA, load the model in FP16 directly (No quantization)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")
print(f"Memory footprint: {model.get_memory_footprint()/1024**3:.2f} GB")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

Total parameters: 3085.9 M
Memory footprint: 5.75 GB


## **3. Define a prompt format and test the BASE model**

In [5]:
def build_prompt(schema: str, question: str) -> str:
    system = "You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else."
    user = f"Schema:\n{schema}\n\nQuestion: {question}"
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

test_prompt_1 = build_prompt(
    schema="CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
    question="List the names of employees in the Engineering department earning more than 100000."
)

print(test_prompt_1)

<|im_start|>system
You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.<|im_end|>
<|im_start|>user
Schema:
CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);

Question: List the names of employees in the Engineering department earning more than 100000.<|im_end|>
<|im_start|>assistant



In [6]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 120) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Slice only newly generated tokens
    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [7]:
output = generate(test_prompt_1)
print(output)

SELECT name FROM employees WHERE department = 'Engineering' AND salary > 100000;


In [8]:
# Three probe prompts we will reuse AFTER fine-tuning for direct comparison.
PROBES = [
    {
        "schema":  "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
        "question":"List the names of employees in the Engineering department earning more than 100000.",
    },
    {
        "schema":  "CREATE TABLE orders (order_id INT, customer_id INT, amount FLOAT, order_date DATE);",
        "question":"What is the total order amount per customer in 2024?",
    },
    {
        "schema":  "CREATE TABLE movies (title TEXT, year INT, rating FLOAT, genre TEXT);",
        "question":"Show the top 5 highest rated horror movies released after 2015.",
    },
]

print("="*70)
print("BASE MODEL (before fine-tuning)")
print("="*70)
base_outputs = []
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    base_outputs.append(ans)
    print(f"\n--- Probe {i} ---")
    print("Q:", p["question"])
    print("A:", ans)


BASE MODEL (before fine-tuning)

--- Probe 1 ---
Q: List the names of employees in the Engineering department earning more than 100000.
A: SELECT name FROM employees WHERE department = 'Engineering' AND salary > 100000;

--- Probe 2 ---
Q: What is the total order amount per customer in 2024?
A: SELECT customer_id, SUM(amount) AS total_order_amount
FROM orders
WHERE YEAR(order_date) = 2024
GROUP BY customer_id;

--- Probe 3 ---
Q: Show the top 5 highest rated horror movies released after 2015.
A: SELECT title, rating
FROM movies
WHERE genre = 'Horror' AND year > 2015
ORDER BY rating DESC
LIMIT 5;


Expect the base model to *talk about* the query, add commentary, re-explain the schema, or produce malformed SQL. That is the "before" state.

## **4. Load and prepare the dataset**

We use `b-mc2/sql-create-context` — a compact text-to-SQL dataset with `(question, context, answer)` triples. We'll take a small slice so training finishes in a few minutes on T4.

In [9]:
from datasets import load_dataset

raw = load_dataset("b-mc2/sql-create-context", split="train")
print("Full dataset size:", len(raw))
print("Example row     :", raw[0])

# Keep it small for a fast, visible demo on T4
raw = raw.shuffle(seed=42).select(range(TRAIN_EXAMPLES))
split = raw.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print("Train:", len(train_ds), " Eval:", len(eval_ds))

README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Full dataset size: 78577
Example row     : {'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}
Train: 4750  Eval: 250


In [10]:
train_ds[:3]

{'answer': ['SELECT COUNT(economy) FROM table_28846752_13 WHERE bbi = "2/19"',
  'SELECT county FROM table_22815568_2 WHERE unemployment_rate = "3.6%"',
  'SELECT score_in_the_final FROM table_26202788_7 WHERE championship = "Miami , United States"'],
 'question': ['How many economy stats for the player with 2/19 BBI?',
  'Which county had a 3.6% unemployment rate?',
  'In the championship Miami , United States, what is the score in the final?'],
 'context': ['CREATE TABLE table_28846752_13 (economy VARCHAR, bbi VARCHAR)',
  'CREATE TABLE table_22815568_2 (county VARCHAR, unemployment_rate VARCHAR)',
  'CREATE TABLE table_26202788_7 (score_in_the_final VARCHAR, championship VARCHAR)']}

In [11]:
print("Schema:", train_ds[0]["context"])
print("="*70)
print("Question:", train_ds[0]["question"])
print("="*70)
print("Answer:", train_ds[0]["answer"])
print("="*70)

Schema: CREATE TABLE table_28846752_13 (economy VARCHAR, bbi VARCHAR)
Question: How many economy stats for the player with 2/19 BBI?
Answer: SELECT COUNT(economy) FROM table_28846752_13 WHERE bbi = "2/19"


In [12]:
def format_example(row):
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user      = f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"
    assistant = row["answer"]
    messages = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": assistant},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map(format_example, remove_columns=eval_ds.column_names)

print("\n--- Formatted training example ---\n")
print(train_ds[0]["text"][:800])# underatand the dataset
train_ds[0].keys()

Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]


--- Formatted training example ---

<|im_start|>system
You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.<|im_end|>
<|im_start|>user
Schema:
CREATE TABLE table_28846752_13 (economy VARCHAR, bbi VARCHAR)

Question: How many economy stats for the player with 2/19 BBI?<|im_end|>
<|im_start|>assistant
SELECT COUNT(economy) FROM table_28846752_13 WHERE bbi = "2/19"<|im_end|>



dict_keys(['text'])

In [13]:
print(train_ds[0])

{'text': '<|im_start|>system\nYou are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.<|im_end|>\n<|im_start|>user\nSchema:\nCREATE TABLE table_28846752_13 (economy VARCHAR, bbi VARCHAR)\n\nQuestion: How many economy stats for the player with 2/19 BBI?<|im_end|>\n<|im_start|>assistant\nSELECT COUNT(economy) FROM table_28846752_13 WHERE bbi = "2/19"<|im_end|>\n'}


## **5. Attach LoRA Adapters**

In [14]:
from peft import LoraConfig, get_peft_model

# Enable gradient checkpointing to save VRAM during backward pass
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# Step 3: Configure QLoRA (LoRA on quantized model)
lora_config = LoraConfig(
    r=16,                              # LoRA rank
    lora_alpha=32,                     # LoRA scaling factor
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],  # Qwen uses q_proj and v_proj
    modules_to_save=["lm_head"],       # Save lm_head as trainable
)

# Step 4: Apply QLoRA to the quantized model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Optional: Print model config for verification
print("\n=== Model Configuration ===")
print(f"Quantization: 4-bit NF4")
print(f"Compute dtype: float16")
print(f"LoRA rank (r): {lora_config.r}")
print(f"LoRA alpha: {lora_config.lora_alpha}")
print(f"Target modules: {lora_config.target_modules}")

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


trainable params: 314,851,328 || all params: 3,400,790,016 || trainable%: 9.2582

=== Model Configuration ===
Quantization: 4-bit NF4
Compute dtype: float16
LoRA rank (r): 16
LoRA alpha: 32
Target modules: {'q_proj', 'v_proj'}


## **6. Train**
We use `SFTTrainer` from TRL.

Settings chosen for T4 (16 GB):
- `per_device_train_batch_size=2`, `gradient_accumulation_steps=8` → effective batch 16
- `max_seq_length=512`
- `fp16=True` (T4 supports FP16, not BF16)
- 1 epoch over 3000 examples ≈ ~5–8 minutes


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
from trl import SFTTrainer, SFTConfig
import torch

OUTPUT_DIR = "/content/drive/MyDrive/qwen-sql-lora-adapters"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEV_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEV_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    gradient_checkpointing=True,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="paged_adamw_32bit",
    # Setting fp16=False here to disable the problematic Accelerator scaler,
    # but the model remains in FP16 from the loading stage.
    fp16=False,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    report_to="none",
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map(tokenize, batched=True, remove_columns=["text"])

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
)

trainer.train()

Step,Training Loss,Validation Loss
50,0.779047,0.794471
100,0.742025,0.739439
150,0.709962,0.715009
200,0.663011,0.700951
250,0.657481,0.688823
300,0.584438,0.675574
350,0.511449,0.692662
400,0.500145,0.690054
450,0.509428,0.683964
500,0.494914,0.682162


TrainOutput(global_step=891, training_loss=0.5641404639858456, metrics={'train_runtime': 2968.2159, 'train_samples_per_second': 4.801, 'train_steps_per_second': 0.3, 'total_flos': 3.228738547212288e+16, 'train_loss': 0.5641404639858456})

In [20]:
# Peak VRAM used during training — should stay under T4's 15.8 GB
import torch
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

Peak GPU memory allocated: 8.73 GB


## **Save MODEL**

In [21]:
# ==================== SAVE FINAL MODEL ====================
trainer.save_model(OUTPUT_DIR)

## **7. Compare: AFTER Fine-Tuning**
You should now see:
- **Before:** verbose, chatty, often wrong syntax, re-explains the schema.
- **After:** terse, well-formed SQL — the model learned the response *style* of the dataset.

That style shift is exactly what LoRA buys you for a few minutes of T4 compute.

In [ ]:
# Re-enable cache for fast inference
model.config.use_cache = True
model.eval()

print("="*70)
print("FINE-TUNED MODEL (after LoRA)")
print("="*70)
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    print(f"\n--- Probe {i} ---")
    print("Q:     ", p["question"])
    print("BEFORE:", base_outputs[i-1])
    print("="*70)
    print("AFTER :", ans)
    print("="*70)

FINE-TUNED MODEL (after LoRA)

--- Probe 1 ---
Q:      List the names of employees in the Engineering department earning more than 100000.
BEFORE: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.
AFTER : SELECT COUNT(name) FROM employees WHERE department = "Engineering" AND salary > 100000

--- Probe 2 ---
Q:      What is the total order amount per customer in 2024?
BEFORE: To answer the question, you need to use the `SUM` function to calculate the total order amount per customer in 2024. Here's the SQL query:

```sql
SELECT SUM(amount) AS total_order_amount_per_customer
FROM orders
WHERE year(order_date) = 2024
GROUP BY customer_id;
```

This query will return a single row with the total order amount per customer in 2024.
AFTER : SELECT SUM(amount) FROM orders WH

## **8. Save the Adapters**
LoRA adapters are tiny (a few MB). You save *only* the adapter, not the whole
base model.

In [22]:
from google.colab import drive
drive.mount('/content/drive')

ADAPTER_DIR = "/content/drive/MyDrive/qwen-sql-lora-adapters"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


('/content/drive/MyDrive/qwen-sql-lora-adapters/tokenizer_config.json',
 '/content/drive/MyDrive/qwen-sql-lora-adapters/chat_template.jinja',
 '/content/drive/MyDrive/qwen-sql-lora-adapters/tokenizer.json')

In [23]:
# temporary dir saving
# ADAPTER_DIR = "./tinyllama-sql-lora-adapter"
# model.save_pretrained(ADAPTER_DIR)
# tokenizer.save_pretrained(ADAPTER_DIR)

import os
total = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Adapter size on disk: {total/1024**2:.2f} MB")

Adapter size on disk: 618.48 MB


## **Loading the adapter later (for reference)**

```python
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

ADAPTER_DIR = "/content/drive/MyDrive/qwen-sql-lora-adapters"

base = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
```


## Recap

| Step | What happened |
|---|---|
| Base model loaded in FP16 | ~2.2 GB VRAM |
| LoRA adapters attached (r=16, target = attention projections) | ~4M trainable params |
| Trained 1 epoch on 3000 SQL examples | ~5–8 min on T4, peak VRAM well under 16 GB |
| Adapter saved | A few MB on disk |
| Visible outcome | Chatty base → clean SQL after fine-tuning |

**To extend this:** swap the dataset (e.g. `databricks/databricks-dolly-15k`, a style-transfer set, or your own JSONL), keep the same LoRA config, and you've got the same workflow for any instruction-following task that fits in T4.


---

**Note on accuracy**

This notebook is a demonstration of the LoRA workflow, not a production SQL model. With a small 1.1B base, 3,000 examples, and 1 epoch, the fine-tuned model reliably learns the response style (clean SQL instead of chatty explanations) but is not always semantically correct. Style learning ≠ task mastery.
To improve accuracy, the same workflow scales up: use a larger base model (e.g. Qwen2.5-3B, or a 7B with QLoRA), more training data (the full dataset has ~78k examples), and 2–3 epochs instead of 1. The code stays the same; only the scale changes.

---